<a href="https://colab.research.google.com/github/neha-lokireddy/BDA-Assignment/blob/main/bda_assignment_q1_083.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Spark Classification Model

### 1. Setup PySpark

In [ ]:
# Install PySpark
!pip install pyspark

# Import SparkSession
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkClassification").getOrCreate()
print("SparkSession initialized.")

SparkSession initialized.


### 2. Load Dataset

I will use the Iris dataset from `sklearn.datasets` for this example. First, load it into a Pandas DataFrame, then convert it to a Spark DataFrame.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

# Load Iris dataset
iris = load_iris()
df_pd = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df_pd['target'] = iris.target

# Convert Pandas DataFrame to Spark DataFrame
df_spark = spark.createDataFrame(df_pd)

print("Schema of Spark DataFrame:")
df_spark.printSchema()

print("First 5 rows of Spark DataFrame:")
df_spark.show(5)

Schema of Spark DataFrame:
root
 |-- sepal length (cm): double (nullable = true)
 |-- sepal width (cm): double (nullable = true)
 |-- petal length (cm): double (nullable = true)
 |-- petal width (cm): double (nullable = true)
 |-- target: long (nullable = true)

First 5 rows of Spark DataFrame:
+-----------------+----------------+-----------------+----------------+------+
|sepal length (cm)|sepal width (cm)|petal length (cm)|petal width (cm)|target|
+-----------------+----------------+-----------------+----------------+------+
|              5.1|             3.5|              1.4|             0.2|     0|
|              4.9|             3.0|              1.4|             0.2|     0|
|              4.7|             3.2|              1.3|             0.2|     0|
|              4.6|             3.1|              1.5|             0.2|     0|
|              5.0|             3.6|              1.4|             0.2|     0|
+-----------------+----------------+-----------------+----------------+-

### 3. Data Preparation

We need to assemble the feature columns into a single vector column using `VectorAssembler` and then split the data into training and test sets.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vectors

# Define feature columns and target column
feature_cols = iris.feature_names
label_col = 'target'

# Assemble features into a single vector column
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_spark_assembled = assembler.transform(df_spark)

# Select only the features and label columns
data = df_spark_assembled.select("features", label_col)

# Split the data into training and test sets (70% train, 30% test)
train_data, test_data = data.randomSplit([0.7, 0.3], seed=42)

print("Training data count:", train_data.count())
print("Test data count:", test_data.count())

print("Sample of prepared data:")
data.show(5, truncate=False)

Training data count: 94
Test data count: 56
Sample of prepared data:
+-----------------+------+
|features         |target|
+-----------------+------+
|[5.1,3.5,1.4,0.2]|0     |
|[4.9,3.0,1.4,0.2]|0     |
|[4.7,3.2,1.3,0.2]|0     |
|[4.6,3.1,1.5,0.2]|0     |
|[5.0,3.6,1.4,0.2]|0     |
+-----------------+------+
only showing top 5 rows


### 4. Model Training (Logistic Regression)

Now, we'll train a Logistic Regression model using the prepared training data.

In [ ]:
from pyspark.ml.classification import LogisticRegression

# Create a Logistic Regression model instance
lr = LogisticRegression(labelCol=label_col, featuresCol="features")

# Train the model
lr_model = lr.fit(train_data)

print("Logistic Regression model trained.")

Logistic Regression model trained.


### 5. Model Evaluation

We will evaluate the model's performance on the test set using `MulticlassClassificationEvaluator` to calculate accuracy.

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Make predictions on the test data
predictions = lr_model.transform(test_data)

# Show sample predictions
print("Sample predictions:")
predictions.select("features", label_col, "prediction", "probability").show(5, truncate=False)

# Evaluate the model
evaluator = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"Test Accuracy = {accuracy}")

Sample predictions:
+-----------------+------+----------+--------------------------------------------------+
|features         |target|prediction|probability                                       |
+-----------------+------+----------+--------------------------------------------------+
|[4.4,3.0,1.3,0.2]|0     |0.0       |[1.0,1.1008475759392545E-29,4.088761019873053E-55]|
|[4.6,3.2,1.4,0.2]|0     |0.0       |[1.0,2.8385525353674E-32,7.127781758785637E-58]   |
|[4.6,3.6,1.0,0.2]|0     |0.0       |[1.0,7.220736349194904E-43,3.1284902865130952E-71]|
|[4.7,3.2,1.3,0.2]|0     |0.0       |[1.0,1.8250774644894864E-32,1.608052600838595E-58]|
|[4.8,3.1,1.6,0.2]|0     |0.0       |[1.0,3.7449823369514343E-28,1.064484763999177E-52]|
+-----------------+------+----------+--------------------------------------------------+
only showing top 5 rows
Test Accuracy = 0.9642857142857143


### 6. Stop SparkSession

It's good practice to stop the SparkSession when you're done.

In [ ]:
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
